<a href="https://colab.research.google.com/github/ArshyaQMUL/diy-covid19dash/blob/Arshya/hackathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clone the repository onto your computer

In [1]:
!git clone https://github.com/ArshyaQMUL/diy-covid19dash.git

Cloning into 'diy-covid19dash'...
remote: Enumerating objects: 102, done.
remote: Counting objects: 100% (65/65), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 102 (delta 50), reused 44 (delta 37), pack-reused 37 (from 1)
Receiving objects: 100% (102/102), 488.91 KiB | 4.94 MiB/s, done.
Resolving deltas: 100% (55/55), done.


Entering the repository folder

In [4]:
%cd /content/diy-covid19dash
!ls

/content/diy-covid19dash
1-Accessing_UKHSA_Data.ipynb	     agedistribution.json  README.md
2-Visualising_the_Data.ipynb	     COPYING		   requirements.txt
3-Adding_Interactive_Controls.ipynb  Dashboard.ipynb	   timeseries.json


recognise my own branch

In [6]:
%cd /content/diy-covid19dash
!git fetch origin

/content/diy-covid19dash
From https://github.com/ArshyaQMUL/diy-covid19dash
 * [new branch]      Arshya     -> origin/Arshya
 * [new branch]      Syed       -> origin/Syed


moving to my own branch

In [8]:
!git switch --track origin/Arshya
!git branch

fatal: a branch named 'Arshya' already exists
* Arshya
  main


### Inspecting the COVID dataset

- `json.load()` reads the JSON file.
- `pd.DataFrame()` converts the data into a table.
- `df.head()` shows the first five rows.
- `df.shape` returns the number of rows and columns.

In [9]:
import json
import pandas as pd

with open("timeseries.json", "r") as file:
    raw_data = json.load(file)

df = pd.DataFrame(raw_data["data"])

display(df.head())
print(df.shape)

,date,cases,hospital,deaths
0,2023-10-14,21.4,NaN,NaN
1,2023-10-13,22.5,7.0,NaN
2,2023-10-12,23.7,7.1,NaN
3,2023-10-11,25.0,7.4,NaN
4,2023-10-10,26.0,7.6,NaN


(1354, 4)


### Step 2: Preparing the DataFrame for visualisation

**Objective:** Prepare the COVID-19 dataset for time-series analysis and plotting.

- `pd.to_datetime()` converts date strings into datetime objects, allowing Pandas to recognise and work with dates.
- `df.sort_values("date")` sorts observations chronologically from oldest to newest.
- `df.set_index("date")` uses dates as row labels, making time-series analysis easier.
- `df.head()` displays the first five observations to inspect the processed data.
- `df.dtypes` checks the data types of the remaining columns.

**Why is this important?**

Time-series analysis requires correctly formatted dates and chronological ordering. Preparing the DataFrame allows us to create graphs showing how COVID-19 statistics changed over time.

**Note:** Missing values (`NaN`) have been preserved because missing measurements do not necessarily represent zero.

In [19]:

# Recreate the DataFrame from our original JSON data
df = pd.DataFrame(raw_data["data"])

# Convert dates into datetime objects
df["date"] = pd.to_datetime(df["date"])

# Sort observations from oldest to newest
df = df.sort_values("date")

# Set the date as the DataFrame index
df = df.set_index("date")

# Inspect our processed data
display(df.head())

# Check column data types
print(df.dtypes)


,cases,hospital,deaths
date,,,
2020-01-30,NaN,NaN,1.0
2020-01-31,NaN,NaN,0.0
2020-02-01,NaN,NaN,0.0
2020-02-02,NaN,NaN,1.0
2020-02-03,NaN,NaN,0.0


cases       float64
hospital    float64
deaths      float64
dtype: object


### Step 2.1: Investigating missing values

**Objective:** Identify and understand missing values in the COVID-19 dataset.

- `NaN` represents missing numerical data in Pandas.
- `df.isna()` identifies missing observations.
- `df.isna().sum()` counts missing values in each column.
- `df.isna().mean() * 100` calculates the percentage of missing data.
- `first_valid_index()` identifies the earliest available observation for each statistic.

**Findings:**

- Cases: 29 missing observations (2.1%).
- Hospitalisations: 56 missing observations (4.1%).
- Deaths: 22 missing observations (1.6%).

**Why is this important?**

Missing data should not automatically be replaced with zero because a missing observation does not necessarily mean the measured value was zero. Incorrectly handling missing values could bias our analysis and produce misleading graphs.

**Decision:** Preserve missing observations for now and allow Pandas to handle gaps during visualisation.

In [20]:

# Count missing values in each column
print(df.isna().sum())

# Calculate the percentage of missing values
print(df.isna().mean() * 100)

# Find the first available date for each statistic
for column in df.columns:
    print(column, df[column].first_valid_index())

cases       29
hospital    56
deaths      22
dtype: int64
cases       2.141802
hospital    4.135894
deaths      1.624815
dtype: float64
cases 2020-02-28 00:00:00
hospital 2020-03-25 00:00:00
deaths 2020-01-30 00:00:00


### Step 3: Creating an interactive COVID-19 graph

**Objective:** Build an interactive time-series graph that allows users to switch between COVID cases, hospitalisations and deaths.

**Libraries used:**
- `ipywidgets`: Creates interactive controls such as dropdowns, sliders and buttons.
- `matplotlib.pyplot`: Generates and customises graphs.
- `IPython.display`: Displays interactive widgets and their outputs in Jupyter notebooks.

**Key functions:**
- `widgets.Dropdown()` creates a dropdown menu containing the available statistics.
- `def plot_metric(metric)` defines a reusable function that plots the selected statistic.
- `df[metric].plot()` selects a DataFrame column and plots its values against the date index.
- `widgets.interactive_output()` connects the dropdown menu to the plotting function.
- `display()` displays the interactive controls and graph.

**How it works:**

When the user selects a different statistic, the plotting function automatically executes again using the newly selected value.

**Why is this important?**

Interactive graphs allow users to explore different datasets without writing or modifying Python code. This makes our dashboard more accessible and useful for analysing public-health trends.

**Data handling:** Missing values remain unchanged. The graph should preserve gaps where observations are unavailable.

In [21]:

import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display

# Create a dropdown menu
metric_dropdown = widgets.Dropdown(
    options=[
        ("Cases", "cases"),
        ("Hospitalisations", "hospital"),
        ("Deaths", "deaths")
    ],
    value="cases",
    description="Statistic:"
)

# Define a function that creates a graph
def plot_metric(metric):
    fig, ax = plt.subplots(figsize=(10, 4))

    df[metric].plot(ax=ax)

    ax.set_title(f"COVID-19: {metric.capitalize()} over time")
    ax.set_xlabel("Date")
    ax.set_ylabel("Metric value (units to verify)")
    ax.grid(True, alpha=0.3)

    plt.show()

# Connect the dropdown menu to the graph function
graph = widgets.interactive_output(
    plot_metric,
    {"metric": metric_dropdown}
)

# Display the dropdown and graph
display(metric_dropdown, graph)


Dropdown(description='Statistic:', options=(('Cases', 'cases'), ('Hospitalisations', 'hospital'), ('Deaths', '…

Output()

### Step 3.1: Interpreting COVID-19 trends

**Observation:** Reported COVID-19 cases appear to peak around January 2022, whereas hospitalisations and deaths show larger peaks around January 2021.

**Possible explanations:**
- COVID-19 vaccination reduced the risk of severe outcomes.
- Omicron became dominant in late 2021 and generally caused less severe disease than earlier variants.
- Immunity from previous infections and vaccination increased over time.
- Changes in testing and reporting affected recorded case numbers.
- Changes in the population's susceptibility may also have contributed.

**Scientific limitation:** These observations are correlations and cannot establish causation without accounting for confounding variables, population characteristics and changes in reporting.

**Conclusion:** The relationship between reported COVID-19 cases and severe outcomes changed substantially during the pandemic, demonstrating why case numbers alone are insufficient to assess disease burden.

### Step 4: Adding interactive date-range filtering

**Objective:** Allow users to select a specific period of the COVID-19 pandemic to visualise.

**New functions and concepts:**

- `pd.date_range()` generates a sequence of dates between the earliest and latest observations.
- `freq="MS"` specifies the start of each month.
- `widgets.SelectionRangeSlider()` creates a slider that allows users to select two dates representing a range.
- `pd.offsets.MonthEnd(0)` determines the final day of the selected ending month.
- `df.loc[start_date:end_date, metric]` filters the DataFrame by date and selected statistic.
- `widgets.interactive_output()` connects multiple interactive controls to the plotting function.

**How it works:**

1. The user selects a COVID-19 statistic from the dropdown.
2. The user selects the starting and ending months using the slider.
3. Python filters the DataFrame to the selected period.
4. Matplotlib generates an updated graph using only the filtered observations.

**Why is this important?**

Date-range filtering enables users to examine specific periods of the pandemic, compare infection waves and investigate changes in disease burden without manually modifying the code.

**Example application:** Comparing the January 2021 and January 2022 COVID-19 waves to explore differences between reported cases, hospitalisations and deaths.

**Scientific consideration:** Missing observations remain unchanged, and trends should be interpreted cautiously because reported COVID statistics may be influenced by testing practices, reporting changes and other confounding factors.

In [22]:

# Create a list of available months from the dataset
months = pd.date_range(
    df.index.min().to_period("M").to_timestamp(),
    df.index.max().to_period("M").to_timestamp(),
    freq="MS"
)

# Create a slider for selecting a range of months
date_slider = widgets.SelectionRangeSlider(
    options=[(date.strftime("%b %Y"), date) for date in months],
    index=(0, len(months)-1),
    description="Date range:",
    continuous_update=False,
    layout=widgets.Layout(width="80%")
)

# Define our improved plotting function
def plot_metric_range(metric, date_range):

    # Extract the start and end months
    start_date, end_month = date_range
    end_date = end_month + pd.offsets.MonthEnd(0)

    # Filter the DataFrame to the selected period
    filtered_data = df.loc[start_date:end_date, metric]

    # Create our graph
    fig, ax = plt.subplots(figsize=(10, 4))

    filtered_data.plot(ax=ax)

    ax.set_title(f"COVID-19: {metric.capitalize()} over time")
    ax.set_xlabel("Date")
    ax.set_ylabel("Metric value (units to verify)")
    ax.grid(True, alpha=0.3)

    plt.show()

# Connect both widgets to our function
interactive_graph = widgets.interactive_output(
    plot_metric_range,
    {
        "metric": metric_dropdown,
        "date_range": date_slider
    }
)

# Display our interactive dashboard
display(metric_dropdown, date_slider, interactive_graph)


Dropdown(description='Statistic:', index=1, options=(('Cases', 'cases'), ('Hospitalisations', 'hospital'), ('D…

SelectionRangeSlider(continuous_update=False, description='Date range:', index=(0, 45), layout=Layout(width='8…

Output()

### Step 5: Multiple statistics and logarithmic scales

**Objective:** Improve the interactive COVID dashboard by allowing users to compare several statistics and customise the graph's Y-axis.

**New features:**

- `widgets.SelectMultiple()` allows multiple statistics to be selected simultaneously.
- `widgets.RadioButtons()` lets users switch between linear and logarithmic scales.
- `df.loc[start_date:end_date, list(metrics)]` filters the dataset by both date range and selected statistics.
- `selected_data.plot()` plots the selected columns together.
- `logy=True` changes the Y-axis to a logarithmic scale.

**Linear vs logarithmic scales:**

- Linear scales display values proportionally to their absolute magnitude.
- Logarithmic scales are useful when values span several orders of magnitude.
- Zero and negative values cannot be plotted on a logarithmic axis, so they are excluded from the logarithmic view.

**Why is this important?**

Combining multiple interactive controls allows users to explore COVID-19 trends without changing the underlying code. The dashboard becomes more flexible and useful for comparing changes over time.

**Scientific considerations:**

- Missing values remain unmodified.
- Different metrics may have different units or definitions, so direct comparisons require caution.
- Correlation between cases and severe outcomes does not establish causation.

In [24]:

# Create a selector that allows multiple statistics
metric_selector = widgets.SelectMultiple(
    options=[
        ("Cases", "cases"),
        ("Hospitalisations", "hospital"),
        ("Deaths", "deaths")
    ],
    value=("cases", "hospital"),
    description="Statistics:",
    rows=3
)

# Create a selector for the graph scale
scale_selector = widgets.RadioButtons(
    options=["linear", "log"],
    value="linear",
    description="Y-axis:"
)

# Create a separate date slider for this new graph
range_selector = widgets.SelectionRangeSlider(
    options=date_slider.options,
    index=date_slider.index,
    description="Date range:",
    continuous_update=False,
    layout=widgets.Layout(width="80%")
)

# Function to plot selected statistics
def plot_multiple(metrics, scale, date_range):

    # Extract the start and end dates
    start_date, end_month = date_range
    end_date = end_month + pd.offsets.MonthEnd(0)

    # Check that at least one statistic is selected
    if len(metrics) == 0:
        print("Please select at least one statistic.")
        return

    # Filter dates and selected columns
    selected_data = df.loc[start_date:end_date, list(metrics)]

    # Logarithmic scales cannot display zero or negative values
    if scale == "log":
        selected_data = selected_data.where(selected_data > 0)

    # Create the graph
    fig, ax = plt.subplots(figsize=(11, 5))

    selected_data.plot(ax=ax, logy=(scale == "log"))

    ax.set_title("COVID-19 Trends Over Time")
    ax.set_xlabel("Date")
    ax.set_ylabel("Metric value (units to verify)")
    ax.legend(title="Statistics")
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# Connect all three controls to the graph
multi_graph = widgets.interactive_output(
    plot_multiple,
    {
        "metrics": metric_selector,
        "scale": scale_selector,
        "date_range": range_selector
    }
)

# Display controls and graph
display(metric_selector, scale_selector, range_selector, multi_graph)


SelectMultiple(description='Statistics:', index=(0, 1), options=(('Cases', 'cases'), ('Hospitalisations', 'hos…

RadioButtons(description='Y-axis:', options=('linear', 'log'), value='linear')

SelectionRangeSlider(continuous_update=False, description='Date range:', index=(0, 45), layout=Layout(width='8…

Output()

### Step 6: Saving my work to GitHub

**Objective:** Save my interactive visualisation notebook to my personal development branch within the team's shared GitHub repository.

- **Repository:** `ArshyaQMUL/diy-covid19dash`
- **Branch:** `Arshya`
- **File:** `Arshya_visualisations.ipynb`

**Key concepts:**

- A Git branch allows team members to develop features independently.
- A commit records a version of the project's files.
- Google Colab can save notebooks directly to GitHub by creating a commit.
- Each team member's contributions can later be merged into the main branch.

**Why is this important?**

Using separate branches allows team members to develop different components simultaneously without directly overwriting each other's code.

**Next objective:** Integrate the interactive graphs with the team's data-processing functions and final dashboard.